# Majority baseline and logistic regression

This notebook evaluates next-five-minute direction predictions using chronological expanding windows. Each test block follows its training dates. The scaler is fit within each training fold. Results are predictive comparisons and do not represent a trading strategy.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import pandas_market_calendars as mcal

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(project_root / "python_code"))
from feature_engineering import FEATURE_COLUMNS, build_model_frame, select_scheduled_bars
from model_evaluation import evaluate_expanding_window
data_dir = project_root / "data"


In [2]:
qqq_source = pd.read_parquet(data_dir / "QQQ_clean_5min.parquet")
spy_source = pd.read_parquet(data_dir / "SPY_clean_5min.parquet")
start_date = min(qqq_source["datetime"].min(), spy_source["datetime"].min()).date()
end_date = max(qqq_source["datetime"].max(), spy_source["datetime"].max()).date()
schedule = mcal.get_calendar("NYSE").schedule(start_date=start_date, end_date=end_date)
qqq_model = build_model_frame(select_scheduled_bars(qqq_source, schedule))
spy_model = build_model_frame(select_scheduled_bars(spy_source, schedule))
assert set(FEATURE_COLUMNS).isdisjoint({"next_datetime", "next_close", "next_return", "target"})
print("Feature columns:", list(FEATURE_COLUMNS))


Feature columns: ['return_1_bar', 'return_3_bars', 'return_6_bars', 'realized_volatility_12_bars', 'bar_range_pct', 'close_location_in_bar', 'log_volume', 'log_volume_change_1_bar', 'minutes_since_open', 'time_of_day_sin', 'time_of_day_cos']


In [3]:
evaluation = {}
for name, frame in [("QQQ", qqq_model), ("SPY", spy_model)]:
    metrics, predictions = evaluate_expanding_window(
        frame, FEATURE_COLUMNS, initial_train_fraction=0.60, n_splits=5
    )
    metrics.insert(0, "ETF", name)
    predictions.insert(0, "ETF", name)
    evaluation[name] = {"metrics": metrics, "predictions": predictions}
    print(f"{name}: {len(metrics['fold'].unique())} folds; {len(predictions):,} out-of-sample rows")
    print(metrics[["fold", "model", "train_start", "train_end", "test_start", "test_end", "train_rows", "test_rows"]].to_string(index=False))


QQQ: 5 folds; 29,012 out-of-sample rows
 fold               model train_start  train_end test_start   test_end  train_rows  test_rows
    1   Majority baseline  2022-03-07 2024-12-06 2024-12-09 2025-04-25       43169       5770
    1 Logistic regression  2022-03-07 2024-12-06 2024-12-09 2025-04-25       43169       5770
    2   Majority baseline  2022-03-07 2025-04-25 2025-04-28 2025-09-09       48939       5819
    2 Logistic regression  2022-03-07 2025-04-25 2025-04-28 2025-09-09       48939       5819
    3   Majority baseline  2022-03-07 2025-09-09 2025-09-10 2026-01-22       54758       5786
    3 Logistic regression  2022-03-07 2025-09-09 2025-09-10 2026-01-22       54758       5786
    4   Majority baseline  2022-03-07 2026-01-22 2026-01-23 2026-06-05       60544       5822
    4 Logistic regression  2022-03-07 2026-01-22 2026-01-23 2026-06-05       60544       5822
    5   Majority baseline  2022-03-07 2026-06-05 2026-06-08 2026-10-16       66366       5815
    5 Logistic regre

SPY: 5 folds; 29,099 out-of-sample rows
 fold               model train_start  train_end test_start   test_end  train_rows  test_rows
    1   Majority baseline  2022-03-07 2024-12-06 2024-12-09 2025-04-25       43049       5895
    1 Logistic regression  2022-03-07 2024-12-06 2024-12-09 2025-04-25       43049       5895
    2   Majority baseline  2022-03-07 2025-04-25 2025-04-28 2025-09-09       48944       5756
    2 Logistic regression  2022-03-07 2025-04-25 2025-04-28 2025-09-09       48944       5756
    3   Majority baseline  2022-03-07 2025-09-09 2025-09-10 2026-01-22       54700       5754
    3 Logistic regression  2022-03-07 2025-09-09 2025-09-10 2026-01-22       54700       5754
    4   Majority baseline  2022-03-07 2026-01-22 2026-01-23 2026-06-05       60454       5861
    4 Logistic regression  2022-03-07 2026-01-22 2026-01-23 2026-06-05       60454       5861
    5   Majority baseline  2022-03-07 2026-06-05 2026-06-08 2026-10-16       66315       5833
    5 Logistic regre

In [4]:
fold_metrics = pd.concat([item["metrics"] for item in evaluation.values()], ignore_index=True)
display(fold_metrics.groupby(["ETF", "model"])[["accuracy", "balanced_accuracy", "f1_up", "roc_auc"]].agg(["mean", "std"]).round(4))

pooled_rows = []
for name, item in evaluation.items():
    pred = item["predictions"]
    for model_name, column in [("Majority baseline", "baseline_prediction"), ("Logistic regression", "logistic_prediction")]:
        pooled_rows.append({
            "ETF": name, "model": model_name,
            "out_of_sample_rows": len(pred),
            "accuracy": (pred["actual"] == pred[column]).mean(),
            "up_label_rate": (pred["actual"] == 1).mean(),
        })
display(pd.DataFrame(pooled_rows).set_index(["ETF", "model"]).round(4))


accuracy         balanced_accuracy           f1_up  \
                            mean     std              mean     std    mean   
ETF model                                                                    
QQQ Logistic regression   0.5070  0.0121            0.5066  0.0133  0.5948   
    Majority baseline     0.5014  0.0045            0.5000  0.0000  0.6679   
SPY Logistic regression   0.5089  0.0108            0.5090  0.0107  0.5602   
    Majority baseline     0.4994  0.0053            0.5000  0.0000  0.6661   

                                roc_auc          
                            std    mean     std  
ETF model                                        
QQQ Logistic regression  0.0198  0.5107  0.0171  
    Majority baseline    0.0040     NaN     NaN  
SPY Logistic regression  0.0150  0.5118  0.0129  
    Majority baseline    0.0047     NaN     NaN

out_of_sample_rows  accuracy  up_label_rate
ETF model                                                           
QQQ Majority baseline                 29012    0.5014         0.5014
    Logistic regression               29012    0.5070         0.5014
SPY Majority baseline                 29099    0.4994         0.4994
    Logistic regression               29099    0.5090         0.4994

## Evaluation boundary

The reported metrics describe out-of-sample classification performance across the five test blocks. The majority baseline is refit from each training block. The logistic-regression pipeline fits its scaler on training rows only. Transaction costs, trading rules, and profitability are not evaluated here.
